# Exploring databases for the project

Picking databases that fit requirements
The aim is to find two datasets, one that dataset should contain compounds that bind to sodium voltage gated ion channels, the second dataset should contain compounds that bind to serotonine receptors.

### Target selection and dataset construction

The aim is to build two datasets, one made out of compounds tested at voltage-gated sodium channels, one made out of compounds tested at serotonin receptors.

#### Targets used:
* Nav1.1 (CHEMBL1845) and Nav1.6 (CHEMBL5202), influx assays, IC50. 
* 5-HT1B (CHEMBL1898) and 5-HT1D (CHEMBL1983), Ki.

#### Filters:
* assay family was identified from assay description rather than `assay_type`. `assay_type` has been proven unreliable here, due to mislabelling.
* Non-null pChEMBL required.
* Replicates aggregated by median.

#### Outputs:
* `nav_set.csv` (711 compounds)
* `ser_set.csv` (701 compounds)

both inside `data/processed`.

The cells below search ChEMBL for candidate targets and examine what each holds. The exclusions are recorded at the end.

In [1]:
#imports
from pathlib import Path

from chembl_webresource_client.new_client import new_client
import pandas as pd

In [2]:
#setting paths for the notebook

FIGURES = Path('../figures')
PROCESSED = Path('../data/processed')
RAW = Path('../data/raw')
RAW.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

On the first glance CHEMBL2095230 looks promising, since it covers both 5-HT1D and 5-HT1B

In [3]:
#database search for serotonine receptors 5-HT1B/1D

target_1 = new_client.target

receptors_1 = ['P28222', 'P28221', 'P30939']

result = target_1.filter(target_components__accession__in=receptors_1)

columns = ['target_chembl_id', 'pref_name', 'target_type', 'organism', 'tax_id', 'species_group_flag', 'cross_references', 'target_components']

df_1 = pd.DataFrame(data=result,columns=columns)
df_1.drop(columns=['target_components','cross_references', 'tax_id', 'species_group_flag'],inplace=True)
print(df_1)


  target_chembl_id                               pref_name        target_type  \
0       CHEMBL1983         5-hydroxytryptamine receptor 1D     SINGLE PROTEIN   
1       CHEMBL1898         5-hydroxytryptamine receptor 1B     SINGLE PROTEIN   
2       CHEMBL1805         5-hydroxytryptamine receptor 1F     SINGLE PROTEIN   
3    CHEMBL2095230  Serotonin 1 receptors; 5-HT1B & 5-HT1D  SELECTIVITY GROUP   
4    CHEMBL2096904               Serotonin (5-HT) receptor     PROTEIN FAMILY   
5    CHEMBL4523958                           5-HT1A/5-HT1F  SELECTIVITY GROUP   
6    CHEMBL4523959                           5-HT1B/5-HT1F  SELECTIVITY GROUP   
7    CHEMBL4523960                           5-HT1D/5-HT1F  SELECTIVITY GROUP   
8    CHEMBL4524122            Serotonin 1 (5-HT1) receptor     PROTEIN FAMILY   

       organism  
0  Homo sapiens  
1  Homo sapiens  
2  Homo sapiens  
3  Homo sapiens  
4  Homo sapiens  
5  Homo sapiens  
6  Homo sapiens  
7  Homo sapiens  
8  Homo sapiens  


Checking what CHEMBL2095230 actually holds

In [4]:
#further exploration of the target CHEMBL2095230

activity_1 = new_client.activity

result_act_1 = activity_1.filter(target_chembl_id='CHEMBL2095230')

df_act_1 = pd.DataFrame(data=result_act_1)

df_act_1.to_csv(RAW / 'serotonin_group_CHEMBL2095230_raw.csv')

molecule_count = df_act_1['molecule_chembl_id'].nunique()
paper_count = df_act_1['document_chembl_id'].nunique()
measurements = df_act_1['standard_type'].value_counts()

print(f"molecule count: {molecule_count}\n",
      f"papers count: {paper_count}\n",
      f"measurements : {measurements}")



molecule count: 254
 papers count: 21
 measurements : standard_type
Selectivity              218
Ratio                     58
Relative efficacy         29
Selectivity ratio         22
Control                    5
Ratio Ki                   4
Degrees Centigrade         3
Ki                         2
Body temperature drop      2
Body temp drop             1
ED50                       1
Name: count, dtype: int64


Almost all of it is derived selectivity and ratio values, rather than primary measurements. This target is rejected.
Moving onto CHEMBL1983.

For CHEMBL1983, checking measurement types and if any records come from mutant variants.

In [5]:
#further exploration of the target CHEMBL1983

activity_2 = new_client.activity

result_act_2 = activity_2.filter(target_chembl_id='CHEMBL1983')

df_act_2 = pd.DataFrame(data=result_act_2)

df_act_2.to_csv(RAW / '5ht1d_CHEMBL1983_raw.csv')

print([c for c in df_act_2.columns if 'organism' in c or 'assay' in c]) #what we have in our columns

molecule_count = df_act_2['molecule_chembl_id'].nunique()
paper_count = df_act_2['document_chembl_id'].nunique()
measurements = df_act_2['standard_type'].value_counts()
var_mut = df_act_2['assay_variant_mutation'].notna().sum()
var_acc = df_act_2['assay_variant_accession'].notna().sum()

print(f"molecule count: {molecule_count}\n",
      f"papers count: {paper_count}\n",
      f"measurements : {measurements}\n",
      f'mutation: {var_mut}\n',
      f'variant accession: {var_acc}')


['assay_chembl_id', 'assay_description', 'assay_type', 'assay_variant_accession', 'assay_variant_mutation', 'target_organism']
molecule count: 2127
 papers count: 378
 measurements : standard_type
Ki                       1400
Inhibition                618
IC50                      436
Intrinsic activity        201
EC50                      196
Efficacy                   92
p[A50]                     51
Activity                   46
Relative efficacy          36
Selectivity                24
ED50                       15
Response                   12
Kb                          9
Ratio                       6
Emax                        6
pKa                         5
NT                          5
IA                          5
pKD                         4
Log Ki                      4
% of inhibition             4
Kd                          3
kon                         2
k_off                       2
Selectivity ratio           1
pKi                         1
% Ctrl                 

Ki is dominating, no variant mutations.

In [6]:
print(df_act_2['pchembl_value'].notna().sum())

1779


In [7]:
#further exploration of the target CHEMBL1898

activity_3 = new_client.activity

result_act_3 = activity_3.filter(target_chembl_id='CHEMBL1898')

df_act_3 = pd.DataFrame(data=result_act_3)

df_act_3.to_csv(RAW / '5ht1b_CHEMBL1898_raw.csv')

molecule_count = df_act_3['molecule_chembl_id'].nunique()
paper_count = df_act_3['document_chembl_id'].nunique()
measurements = df_act_3['standard_type'].value_counts()
var_mut = df_act_3['assay_variant_mutation'].notna().sum()
var_acc = df_act_3['assay_variant_accession'].notna().sum()

print(f"molecule count: {molecule_count}\n",
      f"papers count: {paper_count}\n",
      f"measurements : {measurements}\n",
      f'mutation: {var_mut}\n',
      f'variant accession: {var_acc}')
      



molecule count: 2322
 papers count: 477
 measurements : standard_type
Ki                                                     1328
Inhibition                                              748
IC50                                                    385
Intrinsic activity                                      190
Activity                                                142
EC50                                                    102
pKB                                                      81
Emax                                                     39
Efficacy                                                 23
Selectivity                                              18
% Ctrl                                                   18
Response                                                 12
NT                                                       10
Antagonism                                               10
Kb                                                        9
Simulation                    

This dataset would be similair to the previous target CHEMBL1983 and can be used together with it.

In [8]:
#overlap between last two datasets, how much compounds we have in them
set_1898 = set(df_act_3['molecule_chembl_id'])
set_1983 = set(df_act_2['molecule_chembl_id'])
len(set_1898 & set_1983)

1782

In [9]:
#how much unique compounds we have between two sets
ki_1898 = df_act_3[(df_act_3['standard_type'] == 'Ki') & df_act_3['pchembl_value'].notna()]
ki_1983 = df_act_2[(df_act_2['standard_type'] == 'Ki') & df_act_2['pchembl_value'].notna()]
print(len(set(ki_1898['molecule_chembl_id']) & set(ki_1983['molecule_chembl_id'])))

701


In [10]:
print(f"Replication of compounds in set 1898: {len(ki_1898)} and unique compounds : {len(set(ki_1898['molecule_chembl_id']))}")
print(f"Replication of compounds in set 1983: {len(ki_1983)} and unique compounds : {len(set(ki_1983['molecule_chembl_id']))}")

Replication of compounds in set 1898: 976 and unique compounds : 866
Replication of compounds in set 1983: 1160 and unique compounds : 1002


In [11]:
#deviation in multiple enrties
ki_1898 = ki_1898.copy()
ki_1898['pchembl_value'] = pd.to_numeric(ki_1898['pchembl_value'])
ki_1898.groupby('molecule_chembl_id')['pchembl_value'].std().describe()

count    67.000000
mean      0.177232
std       0.287522
min       0.000000
25%       0.000000
50%       0.035355
75%       0.274212
max       1.560449
Name: pchembl_value, dtype: float64

In [12]:
#deviation in multiple enrties
ki_1983 = ki_1983.copy()
ki_1983['pchembl_value'] = pd.to_numeric(ki_1983['pchembl_value'])
ki_1983.groupby('molecule_chembl_id')['pchembl_value'].std().describe()

count    100.000000
mean       0.199275
std        0.271931
min        0.000000
25%        0.000000
50%        0.098995
75%        0.304056
max        1.796051
Name: pchembl_value, dtype: float64

In [13]:
#database search for sodium voltage gated ion channels subtypes I, II and VI

target_2 = new_client.target

receptors_2 = ['P35498', 'Q99250', 'Q9UQD0']

result = target_2.filter(target_components__accession__in=receptors_2)

columns = ['target_chembl_id', 'pref_name', 'target_type', 'organism', 'tax_id', 'species_group_flag', 'cross_references', 'target_components']

df_2 = pd.DataFrame(data=result,columns=columns)
df_2.drop(columns=['target_components','cross_references', 'tax_id', 'species_group_flag'],inplace=True)
print(df_2)

  target_chembl_id                                          pref_name  \
0       CHEMBL1845        Sodium channel protein type 1 subunit alpha   
1       CHEMBL4187        Sodium channel protein type 2 subunit alpha   
2       CHEMBL5202        Sodium channel protein type 8 subunit alpha   
3    CHEMBL2096682  Sodium channel alpha subunits; brain (Types I,...   
4    CHEMBL2331043                       Sodium channel alpha subunit   
5    CHEMBL4523624                  Fibroblast growth factor 14/SCN8A   
6    CHEMBL4523668                                        SCN1A/SCN1B   
7    CHEMBL4523671                                        SCN8A/SCN1B   
8    CHEMBL4523672                                        SCN2A/SCN1B   
9    CHEMBL4630760  Sodium channel protein type 1 subunit alpha/be...   

                   target_type      organism  
0               SINGLE PROTEIN  Homo sapiens  
1               SINGLE PROTEIN  Homo sapiens  
2               SINGLE PROTEIN  Homo sapiens  
3       

In [ ]:
#further exploration of the target CHEMBL2096682

activity_4 = new_client.activity

result_act_4 = activity_4.filter(target_chembl_id='CHEMBL2096682')

df_act_4 = pd.DataFrame(data=result_act_4)

df_act_4.to_csv(RAW / 'nav_brain_I_II_CHEMBL2096682_raw.csv')

molecule_count = df_act_4['molecule_chembl_id'].nunique()
paper_count = df_act_4['document_chembl_id'].nunique()
measurements = df_act_4['standard_type'].value_counts()
type_of_measurements = df_act_4['assay_description'].value_counts()

print(f"molecule count: {molecule_count}\n",
      f"papers count: {paper_count}\n",
      f"measurements : {measurements}\n",
      f'types of measurements: {type_of_measurements}')



molecule count: 212
 papers count: 5
 measurements : standard_type
Inhibition    207
IC50          163
Binding         4
Name: count, dtype: int64
 types of measurements: assay_description
Inhibition of binding of Batrachotoxinin [3H]BTX-B to high affinity sites on voltage dependent sodium channels in a vesicular preparation from guinea pig cerebral cortex at 10 uM     163
Inhibition of binding of Batrachotoxinin [3H]BTX-B to high affinity sites on voltage dependent sodium channels in a vesicular preparation from guinea pig cerebral cortex              115
Inhibition of binding of Batrachotoxinin [3H]BTX-B to high-affinity sites on voltage-dependent sodium channels in a vesicular preparation from guinea pig cerebral cortex at 100 uM     43
Inhibition of [3H]BTX binding to guinea pig voltage-dependent sodium channel                                                                                                            31
Antagonism of batrachotoxin-mediated sodium ion uptake into cul

This dataset would have 207 rows of binding and it has misslabeled organism on some, moving on to next target.

In [15]:
#further exploration of the target CHEMBL1845

activity_5 = new_client.activity

result_act_5 = activity_5.filter(target_chembl_id='CHEMBL1845')

df_act_5 = pd.DataFrame(data=result_act_5)

df_act_5.to_csv(RAW / 'nav11_CHEMBL1845_raw.csv')

molecule_count = df_act_5['molecule_chembl_id'].nunique()
paper_count = df_act_5['document_chembl_id'].nunique()
measurements = df_act_5['standard_type'].value_counts()
type_of_measurements = df_act_5['assay_description'].value_counts()

print(f"molecule count: {molecule_count}\n",
      f"papers count: {paper_count}\n",
      f"measurements : {measurements}\n",
      f'types of measurements: {type_of_measurements}')



molecule count: 1028
 papers count: 54
 measurements : standard_type
IC50           1107
kon            1010
k_off          1010
Inhibition      167
Activity        102
Ratio AUC        72
V1/2             34
delta(V1/2)      32
EC50              7
T/C               6
Ratio IC50        4
Kd                3
Ratio             2
Name: count, dtype: int64
 types of measurements: assay_description
Sodium Influx Assay (In Vitro Assay): This sodium influx assay employs the use of the cell permeable, sodium sensitive dye ANG2 to quantify sodium ion influx through sodium channels which are maintained in an open state by use of sodium channel modulators. This high throughput sodium influx assay allows for rapid profiling and characterization of sodium channel blockers.In general, Trex HEK293 cells were stably transfected with an inducible expression vector containing the full-length cDNA coding for the desired human sodium channel α-subunit and with an expression vector containing full length c

Too many assay descriptions, further filtering needed.

In [16]:
#further exploration of the target CHEMBL1845

activity_5 = new_client.activity

result_act_5 = activity_5.filter(target_chembl_id='CHEMBL1845')

df_act_5 = pd.DataFrame(data=result_act_5)

df_act_5.to_csv(RAW / 'nav11_CHEMBL1845_raw.csv')

molecule_count = df_act_5['molecule_chembl_id'].nunique()
paper_count = df_act_5['document_chembl_id'].nunique()
measurements = df_act_5['standard_type'].value_counts()
type_of_measurements = df_act_5['assay_description'].str.contains('Influx', case=False, na=False).value_counts()

print(f"molecule count: {molecule_count}\n",
      f"papers count: {paper_count}\n",
      f"measurements : {measurements}\n",
      f'Influx and patch clamp rows: {type_of_measurements}')



molecule count: 1028
 papers count: 54
 measurements : standard_type
IC50           1107
kon            1010
k_off          1010
Inhibition      167
Activity        102
Ratio AUC        72
V1/2             34
delta(V1/2)      32
EC50              7
T/C               6
Ratio IC50        4
Kd                3
Ratio             2
Name: count, dtype: int64
 Influx and patch clamp rows: assay_description
True     2497
False    1059
Name: count, dtype: int64


Adding IC50 filter to avoid other measurement results.

In [17]:
#further exploration of the target CHEMBL1845

activity_5 = new_client.activity

result_act_5 = activity_5.filter(target_chembl_id='CHEMBL1845')

df_act_5 = pd.DataFrame(data=result_act_5)

df_act_5.to_csv(RAW / 'nav11_CHEMBL1845_raw.csv')

molecule_count = df_act_5['molecule_chembl_id'].nunique()
paper_count = df_act_5['document_chembl_id'].nunique()
measurements = df_act_5['standard_type'].value_counts()
influx_5 = df_act_5['assay_description'].str.contains('Influx', case=False, na=False)
ic50_5 = df_act_5['standard_type'] == 'IC50'
target_rows_influx = (influx_5 & ic50_5).sum()
influx_compounds = df_act_5.loc[influx_5 & ic50_5, 'molecule_chembl_id'].nunique()
clamp = df_act_5['assay_description'].str.contains('clamp', case=False, na=False)
target_rows_clamp = (clamp & ic50_5).sum()
clamp_compounds = df_act_5.loc[clamp & ic50_5, 'molecule_chembl_id'].nunique()

shared_compunds = len(set(df_act_5.loc[clamp & ic50_5, 'molecule_chembl_id']) & set(df_act_5.loc[influx_5 & ic50_5, 'molecule_chembl_id']))

print(f"molecule count: {molecule_count}\n",
      f"papers count: {paper_count}\n",
      f"measurements : {measurements}\n",
      f'influx: {target_rows_influx} and influx compounds: {influx_compounds}\n',
      f"clamp: {target_rows_clamp} and clamp compounds: {clamp_compounds}\n",
      f"shared compounds: {shared_compunds}")


molecule count: 1028
 papers count: 54
 measurements : standard_type
IC50           1107
kon            1010
k_off          1010
Inhibition      167
Activity        102
Ratio AUC        72
V1/2             34
delta(V1/2)      32
EC50              7
T/C               6
Ratio IC50        4
Kd                3
Ratio             2
Name: count, dtype: int64
 influx: 833 and influx compounds: 780
 clamp: 221 and clamp compounds: 212
 shared compounds: 176


Influx and clamp measurements are counted separetly since mixing assay families would be mixing measurements conditions. Moving on to next target.

In [18]:
#further exploration of the target CHEMBL4187

activity_6 = new_client.activity

result_act_6 = activity_6.filter(target_chembl_id='CHEMBL4187')

df_act_6 = pd.DataFrame(data=result_act_6)

df_act_6.to_csv(RAW / 'nav12_CHEMBL4187_raw.csv')

molecule_count = df_act_6['molecule_chembl_id'].nunique()
paper_count = df_act_6['document_chembl_id'].nunique()
measurements = df_act_6['standard_type'].value_counts()
influx = df_act_6['assay_description'].str.contains('Influx', case=False, na=False)
ic50 = df_act_6['standard_type'] == 'IC50'
target_rows_influx = (influx & ic50).sum()
influx_compounds = df_act_6.loc[influx & ic50, 'molecule_chembl_id'].nunique()
clamp = df_act_6['assay_description'].str.contains('clamp', case=False, na=False)
target_rows_clamp = (clamp & ic50).sum()
clamp_compounds = df_act_6.loc[clamp & ic50, 'molecule_chembl_id'].nunique()

shared_compunds = len(set(df_act_6.loc[clamp & ic50, 'molecule_chembl_id']) & set(df_act_6.loc[influx & ic50, 'molecule_chembl_id']))

print(f"molecule count: {molecule_count}\n",
      f"papers count: {paper_count}\n",
      f"measurements : {measurements}\n",
      f'influx: {target_rows_influx} and influx compounds: {influx_compounds}\n',
      f"clamp: {target_rows_clamp} and clamp compounds: {clamp_compounds}\n",
      f"shared compounds: {shared_compunds}")


molecule count: 341
 papers count: 81
 measurements : standard_type
Inhibition         248
IC50               218
T/C                 99
Activity            33
Ki                  12
Ratio IC50           6
V1/2                 6
EC50                 5
Ratio AUC            4
% of inhibition      4
Kd                   3
% of effect          1
% of control         1
Name: count, dtype: int64
 influx: 1 and influx compounds: 1
 clamp: 92 and clamp compounds: 72
 shared compounds: 1


Moving on to next target.

In [19]:
#further exploration of the target CHEMBL5202

activity_7 = new_client.activity

result_act_7 = activity_7.filter(target_chembl_id='CHEMBL5202')

df_act_7 = pd.DataFrame(data=result_act_7)

df_act_7.to_csv(RAW / 'nav16_CHEMBL5202_raw.csv')

molecule_count = df_act_7['molecule_chembl_id'].nunique()
paper_count = df_act_7['document_chembl_id'].nunique()
measurements = df_act_7['standard_type'].value_counts()
influx_7 = df_act_7['assay_description'].str.contains('Influx', case=False, na=False)
ic50_7 = df_act_7['standard_type'] == 'IC50'
target_rows_influx = (influx_7 & ic50_7).sum()
influx_compounds = df_act_7.loc[influx_7 & ic50_7, 'molecule_chembl_id'].nunique()
clamp = df_act_7['assay_description'].str.contains('clamp', case=False, na=False)
target_rows_clamp = (clamp & ic50_7).sum()
clamp_compounds = df_act_7.loc[clamp & ic50_7, 'molecule_chembl_id'].nunique()

shared_compunds = len(set(df_act_7.loc[clamp & ic50_7, 'molecule_chembl_id']) & set(df_act_7.loc[influx_7 & ic50_7, 'molecule_chembl_id']))

print(f"molecule count: {molecule_count}\n",
      f"papers count: {paper_count}\n",
      f"measurements : {measurements}\n",
      f'influx: {target_rows_influx} and influx compounds: {influx_compounds}\n',
      f"clamp: {target_rows_clamp} and clamp compounds: {clamp_compounds}\n",
      f"shared compounds: {shared_compunds}")

 

molecule count: 1175
 papers count: 63
 measurements : standard_type
IC50           2071
kon            1944
k_off          1944
Inhibition       49
K                20
Activity         18
Kd               12
EC50              9
Delta Tm          7
T/C               6
Ratio IC50        4
V1/2              4
Ratio AUC         4
Kact              1
Kinact            1
TIME              1
delta(V1/2)       1
Name: count, dtype: int64
 influx: 1165 and influx compounds: 805
 clamp: 684 and clamp compounds: 373
 shared compounds: 235


Influx has far more data and will be used for both Nav1.1 and Nav1.6 data on which model will be build.

In [20]:
#overlaping compounds inside Nav channel sets
len(set(df_act_5.loc[influx_5 & ic50_5, 'molecule_chembl_id']) & set(df_act_7.loc[influx_7 & ic50_7, 'molecule_chembl_id']))

780

Every compound inside Nav1.1 set that has pIC50 measurement, also is present inside Nav1.6. That is 780 overlaping compounds. 

In [21]:
#overlaping compounds between two future datasets
set_1845 = set(df_act_5.loc[influx_5 & ic50_5, 'molecule_chembl_id'])
set_5202=set(df_act_7.loc[influx_7 & ic50_7, 'molecule_chembl_id'])

len(set_1845 & set_5202 & set_1898 & set_1983)

# 0 is the best result

0

In [22]:
#Merging Nav sets

prep_df_7 = df_act_7[df_act_7['assay_description'].str.contains('Influx', case=False, na=False) & (df_act_7['standard_type'] == 'IC50')].copy()
prep_df_5 = df_act_5[df_act_5['assay_description'].str.contains('Influx', case=False, na=False) & (df_act_5['standard_type'] == 'IC50')].copy()

prep_df_5 = prep_df_5[prep_df_5['pchembl_value'].notna()]
prep_df_7 = prep_df_7[prep_df_7['pchembl_value'].notna()]

prep_df_5['pchembl_value'] = pd.to_numeric(prep_df_5['pchembl_value'])
prep_df_7['pchembl_value'] = pd.to_numeric(prep_df_7['pchembl_value'])

prep_df_7 = prep_df_7.groupby('molecule_chembl_id').agg({'pchembl_value': 'median', 'canonical_smiles': 'first'})
prep_df_5 = prep_df_5.groupby('molecule_chembl_id').agg({'pchembl_value': 'median', 'canonical_smiles': 'first'})

nav_set = pd.merge(prep_df_5, prep_df_7, how='inner', on='molecule_chembl_id', suffixes=('_nav11', '_nav16'))

nav_set.drop(columns=['canonical_smiles_nav11'],inplace=True)

nav_set.to_csv(PROCESSED / 'nav_set.csv', index=True)

len(nav_set)

711

The merge yielded 711 compounds because it required non-null values for pChEMBL, at both endpoints.

In [23]:
#Merging Serotonine sets

prep_df_2 = df_act_2[df_act_2['standard_type'] == 'Ki'].copy()
prep_df_3 = df_act_3[df_act_3['standard_type'] == 'Ki'].copy()

prep_df_2 = prep_df_2[prep_df_2['pchembl_value'].notna()]
prep_df_3 = prep_df_3[prep_df_3['pchembl_value'].notna()]

prep_df_2['pchembl_value'] = pd.to_numeric(prep_df_2['pchembl_value'])
prep_df_3['pchembl_value'] = pd.to_numeric(prep_df_3['pchembl_value'])

prep_df_2 = prep_df_2.groupby('molecule_chembl_id').agg({'pchembl_value': 'median', 'canonical_smiles': 'first'})
prep_df_3 = prep_df_3.groupby('molecule_chembl_id').agg({'pchembl_value': 'median', 'canonical_smiles': 'first'})

ser_set = pd.merge(prep_df_2, prep_df_3, how='inner', on='molecule_chembl_id', suffixes=('_5ht1b', '_5ht1d'))

ser_set.drop(columns=['canonical_smiles_5ht1b'],inplace=True)

ser_set.to_csv(PROCESSED / 'ser_set.csv', index=True)

len(ser_set)


701

### Targets considered and excluded

Nav:
* CHEMBL2096682 (brain types I/II family) — BTX radioligand binding, guinea pig cerebral cortex, mostly single-point percent inhibition. Wrong measurement type, mixed subtypes. 
* CHEMBL4187 (Nav1.2) — one influx row; the screening programme didn't cover this subtype. Complexes and the FGF14 interaction target excluded as different mechanisms.

5-HT: 
* CHEMBL2095230 (1B/1D selectivity group) — 345 rows, 218 Selectivity + 58 Ratio, 2 raw Ki. Holds derived ratios, not primary measurements.

### Data quality findings

* Replicate spread: 5-HT1B median 0.035 max 1.56 (67 compounds); 5-HT1D median 0.099 max 1.80 (100 compounds).
* ChEMBL records whether measurement is exact or at the limit, but that flag cannot be trusted here. Inside Nav1.1 set there were 143 records marked at 4.52 pIC50, which is standard top screening measurement of 30 µM on these tests. A compound that shows no effect at this concentration, has a true value somewhere below it. The 71 records marked as the limits are a different group for Nav1.1. Nav1.6 has 4 such records. Trusting the flag would mean all 143 records are treated as true measurements.
* Wild-type only: zero variant mutations at CHEMBL1983.

### Structural findings

* Nav1.1 influx compounds are a complete subset of Nav1.6 (780/780). Merge drops it to 711 due to exclusion of pChEMBL null values.
* Patch clamp overlaps influx heavily — 176/212 at Nav1.1, 235/373 at Nav1.6 — so clamp validates measurement conditions, not held-out chemistry.
* Zero cross-side overlap between Nav and 5-HT sets, before any filtering.

### Final counts

711 Nav, 701 5-HT.